In [35]:
import torch
import torchvision
from torchvision.transforms import v2
import torch.nn as nn
import torch.nn.functional as F

In [36]:
batch_size = 32
learning_rate = 1e-3
num_epochs = 15

In [37]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True), #scale normalizes to (0,1) range
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

In [38]:
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

In [39]:
oc_1 = 32
oc_2 = 64
oc_3 = 128
k_size_conv = 3
k_size_pool = 2

In [40]:
class CNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(3, oc_1, k_size_conv, padding=1, stride = 1)
    self.bn1 = nn.BatchNorm2d(oc_1)
    self.conv2 = nn.Conv2d(oc_1, oc_2, k_size_conv, padding=1, stride = 1)
    self.bn2 = nn.BatchNorm2d(oc_2)
    self.conv3 = nn.Conv2d(oc_2, oc_3, k_size_conv, padding=1, stride = 1)
    self.bn3 = nn.BatchNorm2d(oc_3)
    self.pool = nn.MaxPool2d(k_size_pool, 2)
    self.fc1 = nn.Linear(4*4*oc_3, 1024)
    self.fc2 = nn.Linear(1024, 256)
    self.fc3 = nn.Linear(256, 10)

  def forward(self,x):
    x = self.pool(F.relu(self.bn1(self.conv1(x))))
    # (b,3x32x32) -> (b,32x32x32) -> (b,32x16x16)
    x = self.pool(F.relu(self.bn2(self.conv2(x))))
    # (b,32x16x16) -> (b,64x16x16) -> (b,64x8x8)
    x = self.pool(F.relu(self.bn3(self.conv3(x))))
    # (b,64x8x8) -> (b,128x8x8) -> (b,128x4x4)
    x = torch.flatten(x,1)
    # (b,128x4x4) -> (b,2048)
    x = F.relu(self.fc1(x))
    # (b,2048) -> (b,1024)
    x = F.relu(self.fc2(x))
    # (b,1024) -> (b,256)
    x = self.fc3(x)
    # (b,256) -> (b,10)
    return x


In [41]:
model = CNN()

In [42]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()
criterion.to(device)

cpu


CrossEntropyLoss()

In [24]:
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(trainloader)
    print(f'Epoch [{epoch + 1}/{num_epochs}] completed | Loss: {epoch_loss:.4f}')

print('Finished Training')

Epoch [1/15] completed | Loss: 1.2356
Epoch [2/15] completed | Loss: 0.8304
Epoch [3/15] completed | Loss: 0.6821
Epoch [4/15] completed | Loss: 0.5692
Epoch [5/15] completed | Loss: 0.4800
Epoch [6/15] completed | Loss: 0.4015
Epoch [7/15] completed | Loss: 0.3296
Epoch [8/15] completed | Loss: 0.2719
Epoch [9/15] completed | Loss: 0.2203
Epoch [10/15] completed | Loss: 0.1853
Epoch [11/15] completed | Loss: 0.1547
Epoch [12/15] completed | Loss: 0.1304
Epoch [13/15] completed | Loss: 0.1184
Epoch [14/15] completed | Loss: 0.0995
Epoch [15/15] completed | Loss: 0.0959
Finished Training


In [25]:
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')

Accuracy of the network on the 10000 test images: 78 %


## transfer learning


In [26]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

In [27]:
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [28]:
trainset.transform = transform_train
testset.transform = transform_test

trainloader_tl = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
testloader_tl = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

In [29]:
model_tl = resnet18(weights=ResNet18_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 90.3MB/s]


In [30]:
print(model_tl)

for name, module in model_tl.named_children():
    print(name)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [44]:
for param in model_tl.parameters():
    param.requires_grad = False

# retraining the last two layers
# for param in model_tl.layer3.parameters():
#     param.requires_grad = True

for param in model_tl.layer4.parameters():
    param.requires_grad = True

# the final classifier head
for param in model_tl.fc.parameters():
    param.requires_grad = True

In [45]:
num_features = model_tl.fc.in_features
model_tl.fc = nn.Linear(num_features, 10)
model_tl.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [46]:
# optimizer_tl = optim.Adam(model_tl.parameters(), lr=learning_rate)
optimizer_tl = optim.Adam(filter(lambda p: p.requires_grad, model_tl.parameters()), lr=1e-4)

In [ ]:
for epoch in range(num_epochs):
    model_tl.train()
    running_loss = 0.0
    for i, data in enumerate(trainloader_tl, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer_tl.zero_grad()
        outputs = model_tl(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_tl.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(trainloader)
    print(f'Epoch [{epoch + 1}/{num_epochs}] completed | Loss: {epoch_loss:.4f}')

print('Finished Training')

In [ ]:
correct = 0
total = 0
with torch.no_grad():
    for data in testloader_tl:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = model_tl(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')